# 02 — Data Profiling: Raw GeoNames Data

## Purpose

Profile the raw GeoNames HU.txt file prior to transformation.

## Findings Summary
- All GeoNames records contain non-null `name`, `latitude`, and `longitude`

- Latitude and longitude values fall within valid Hungarian geographic bounds

- Significant number of duplicate settlement names detected within the populated-place subset

- Budapest district naming follows multiple inconsistent conventions:
  - Roman numeral format (e.g. `Budapest I.`)
  - Expanded kerület format (e.g. `Budapest I. kerület`)

- The complete set of Budapest districts is represented when using the `Budapest [IVX]+.` naming convention

---


## Setup

In [ ]:
import pandas as pd

from hpm.settings import settings

pd.set_option("display.max_colwidth", None)

GEONAMES_COLUMNS = [
    "geonameid",
    "name",
    "asciiname",
    "alternatenames",
    "latitude",
    "longitude",
    "feature class",
    "feature code",
    "county code",
    "cc2",
    "admin1 code",
    "admin2 code",
    "admin3 code",
    "admin4 code",
    "population",
    "elevation",
    "dem",
    "timezone",
    "modification date",
]

path = settings.raw_settlements / "HU.txt"

df = pd.read_csv(
    path,
    sep="\t",
    index_col=None,
    header=None,
    names=GEONAMES_COLUMNS,
    low_memory=False,
)

## Content Finding 1 — `latitude` and `longitude` Non-Null

In [ ]:
assert df["latitude"].notna().all(), "❌ latitude contains nulls"
assert df["longitude"].notna().all(), "❌ longitude contains nulls"
print("✅ Latitude and longitude are always populated")

## Content Finding 2 — `latitude` and `longitude` in Expected Range

In [ ]:
assert df["latitude"].between(45, 49).all(), "❌ invalid latitude values"

assert df["longitude"].between(16, 23).all(), "❌ invalid longitude values"

print("✅ Coordinates fall within valid geographic ranges")

## Content Finding 3 — `name` Non-Null

In [ ]:
assert df["name"].notna().all(), "❌ latitude contains nulls"
print("✅ name column is always populated")

## Content Finding 4 — Populated Feature Duplicates

In [ ]:
feature_codes = {
    "PPL",
    "PPLA",
    "PPLA2",
    "ADM2",
}

populated = df[df["feature code"].isin(feature_codes)].copy()

dup_mask = populated["name"].duplicated(keep=False)
duplicate_groups = populated[dup_mask].sort_values("name")

n_duplicate_rows = populated["name"].duplicated().sum()
n_duplicate_names = duplicate_groups["name"].nunique()

print(f"Populated-place records: {len(populated)}")
print(f"Duplicate name groups: {n_duplicate_names}")
print(
    f"Excess rows beyond first occurrence: {n_duplicate_rows} "
    f"({n_duplicate_rows / len(populated):.2%} of records)"
)

display(duplicate_groups[["name", "feature code", "latitude", "longitude"]])

## Content Finding 5 — Budapest Districts Variants

In [ ]:
roman = df[df["name"].str.match(r"Budapest [IVX]+\.")]
numeric = df[df["name"].str.match(r"Budapest \d+")]

summary = pd.DataFrame(
    [
        {"format": "Roman numeral", "count": roman.shape[0]},
        {"format": "Numeric", "count": numeric.shape[0]},
    ]
)

display(summary)

## Content Finding 6 — Budapest District Coverage in "kerület" Naming Convention

In [ ]:
expected_districts = 23  # Budapest districts

kerulet = df[df["name"].str.match(r"Budapest [IVX]+\. kerület", na=False)][
    "name"
]

missing_count = kerulet.shape[0]

display(kerulet.to_frame())

print(
    f"Count of 'kerület' format entries: {missing_count}, missing {expected_districts - missing_count}"
)

## Content Finding 7 — Completeness of Budapest District Coverage (Roman Numeral Format)

In [ ]:
complete = df[df["name"].str.match(r"Budapest [IVX]+\.$", na=False)]

display(complete[["name"]])

observed_count = complete.shape[0]
expected_count = 23  # Budapest districts

assert observed_count == expected_count, (
    f"Expected {expected_count} Budapest districts, found {observed_count}"
)

print(
    f"✅ Budapest Roman numeral format is complete ({observed_count}/{expected_count})"
)